In [0]:
!pip install lightgbm
import mlflow
from mlflow import MlflowClient

print("MLflow version:", mlflow.__version__)

In [0]:
CATALOG = "workspace"
SCHEMA = "default"

MODEL_NAME = f"{CATALOG}.{SCHEMA}.demand_forecasting"

EXPERIMENT_NAME = "/Shared/demand-forecasting"

mlflow.set_registry_uri("databricks-uc")

client = MlflowClient()

print("Experiment :", EXPERIMENT_NAME)
print("Model      :", MODEL_NAME)
print("Registry   : databricks-uc")

In [0]:
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    raise ValueError(
        f"Experiment '{EXPERIMENT_NAME}' was not found."
    )

EXPERIMENT_ID = experiment.experiment_id

print("Experiment ID:", EXPERIMENT_ID)

In [0]:
runs = client.search_runs(
    experiment_ids=[EXPERIMENT_ID],
    filter_string="status = 'FINISHED'",
    order_by=["start_time DESC"],
    max_results=20
)

if not runs:
    raise ValueError(
        "No successful MLflow runs found in the experiment."
    )

print(f"Found {len(runs)} completed runs.")

In [0]:
for run in runs:
    print(
        f"Run ID: {run.info.run_id} | "
        f"Run Name: {run.data.tags.get('mlflow.runName')} | "
        f"Start Time: {run.info.start_time}"
    )

In [0]:
latest_run = runs[0]

RUN_ID = latest_run.info.run_id

print("Selected Run ID:", RUN_ID)
print("Run Name:", latest_run.data.tags.get("mlflow.runName"))

In [0]:
print("Metrics from selected run:")

for key, value in latest_run.data.metrics.items():
    print(f"{key}: {value}")

In [0]:
MODEL_URI = f"runs:/{RUN_ID}/model"

print("Model URI:", MODEL_URI)

In [0]:
loaded_model = mlflow.lightgbm.load_model(MODEL_URI)

print("Model loaded successfully.")
print("Model type:", type(loaded_model))

In [0]:
registered_model = mlflow.register_model(model_uri=MODEL_URI, name=MODEL_NAME)

print("Model registered successfully.")
print("Model name   :", registered_model.name)
print("Model version:", registered_model.version)

In [0]:
import time

model_version = registered_model.version

for _ in range(30):
    mv = client.get_model_version(
        name=MODEL_NAME,
        version=model_version
    )

    print("Status:", mv.status)

    if mv.status == "READY":
        print("Model version is READY.")
        break

    time.sleep(2)
else:
    print("Model version is still being processed.")

In [0]:
client.set_model_version_tag(
    name=MODEL_NAME,
    version=model_version,
    key="model_type",
    value="LightGBM"
)

client.set_model_version_tag(
    name=MODEL_NAME,
    version=model_version,
    key="problem_type",
    value="demand_forecasting"
)

client.set_model_version_tag(
    name=MODEL_NAME,
    version=model_version,
    key="framework",
    value="LightGBM"
)

print("Model version tags added.")

In [0]:
client.update_model_version(
    name=MODEL_NAME,
    version=model_version,
    description=(
        "LightGBM demand forecasting model trained using "
        "historical demand lags, rolling statistics, "
        "calendar features, business features, and categorical features."
    )
)

print("Model description updated.")

In [0]:
model_info = client.get_model_version(
    name=MODEL_NAME,
    version=model_version
)

print("====================================")
print("Registered Model")
print("====================================")
print("Name       :", model_info.name)
print("Version    :", model_info.version)
print("Status     :", model_info.status)
print("Run ID     :", model_info.run_id)
print("Source     :", model_info.source)
print("Description:", model_info.description)
print("====================================")

In [0]:
versions = client.search_model_versions(
    filter_string=f"name = '{MODEL_NAME}'"
)

print(f"Registered versions for {MODEL_NAME}:\n")

for version in versions:
    print(
        f"Version: {version.version} | "
        f"Status: {version.status} | "
        f"Run ID: {version.run_id}"
    )